# S6E10 — 03 Final Features (`X_final`)

> **Pipeline position:** `00_eda`, `01_baseline`, `02_features_experiments` → **03 final features** → 04 LightGBM → 05 CatBoost/XGB → 06 MLP → 07 RealMLP → … → 11 stack

| | |
|---|---|
| **Purpose** | Rebuild the *frozen* feature set in one fast run and save it, so every later notebook just loads parquet files instead of repeating experiments. |
| **Input** | `data/raw/train.csv`, `test.csv`, `orig_data.csv` (the original survey the competition data was generated from) |
| **Output** | `X_final_train.parquet`, `X_final_test.parquet`, `y_train.parquet` in `data/processed/` (on Kaggle: `/kaggle/working`) |
| **Runtime** | ~5 min to build + ~3 min for the ablation (section 6); +25–40 min if the optional check in section 7 is switched on |

### What `X_final` consists of (45 columns)
| Block | Columns | Idea |
|---|---|---|
| Baseline | 21 | Original columns (names lower-cased) + `arrival_delay_missing` flag; `arrival_delay` itself is dropped |
| Frequency | 2 | How often a `flight_distance` value (and the `flight_distance`×`type_of_travel` pair) occurs |
| Route profile | 20 `fdm_*` | `flight_distance` acts like a route ID (EDA §6.4), so we describe each route by the mean of all other features |
| Original-data signal | 2 | `orig_prob`, `orig_logit`: a model trained only on the original survey scores each competition row |

`te_flight_distance` (target encoding) is deliberately **not** saved: it uses the label, so it must be computed *inside each CV fold* (helper in the last cell).

### Does each block earn its place?
Section 6 measures it (LightGBM with the baseline parameters, in-fold TE of the distance, the same 5 folds as everywhere):

| Feature set | CV AUC | Gain | Folds improved |
|---|---|---|---|
| 21 baseline + 2 frequency + TE (end of `02`) | 0.96015 | – | – |
| + route profile | 0.96036 | +0.00020 | 5 / 5 |
| + original-data signal | 0.96027 | +0.00011 | 5 / 5 |
| **+ both = `X_final`** | **0.96048** | **+0.00032** | **5 / 5** |

### Reference numbers (the notebook asserts them)
orig-model AUC on train ≈ **0.95472**; tuned LightGBM on `X_final`: CV **0.96072**, LB 0.96036. For comparison, the plain baseline LightGBM scored CV 0.95885 (`01_baseline`, `02_features_experiments`).

> **Glossary for beginners**
> - **OOF (out-of-fold) prediction** – each row is predicted by a model that never saw that row in training. It gives an honest score and the inputs for stacking.
> - **Target leakage** – a feature that secretly contains the answer (e.g. computed from the label on the same rows). It inflates CV and fails on the leaderboard.
> - **Target encoding (TE)** – replace a category by the mean of the target in that category. Powerful, but must be fitted on the training part of a fold only.
> - **Parquet** – compact binary table format; keeps dtypes (including `category`) and loads much faster than CSV.
> - **Ablation** – measure a block's value by adding (or removing) it while everything else stays fixed.

## 1. Setup
**Why:** one place for paths, seed and folds. Paths work on Kaggle and locally (from the project root or from a subfolder). `RANDOM_STATE = 42` and `StratifiedKFold(5, shuffle=True)` are reused by *every* notebook — identical folds are what makes OOF predictions of different models stackable without leakage.

In [1]:
import os, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Path logic: Kaggle input first, then ../data/raw (running from a subfolder), then data/raw.
DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e10') if Path('/kaggle/input/competitions/playground-series-s6e10').exists() \
           else next((p for p in (Path('data/raw'), Path('../data/raw')) if p.exists()), Path('data/raw'))
ORIG_PATH = next((p for p in [Path('/kaggle/input/datasets/deonissx/s6e10-orig-data/orig_data.csv'),
                              Path('data/raw/orig_data.csv'), Path('../data/raw/orig_data.csv')] if p.exists()),
                 Path('data/raw/orig_data.csv'))   # Kaggle path first, then local project folder
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else next((p for p in (Path('data/processed'), Path('../data/processed')) if p.exists()), Path('data/processed'))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Same 5 folds (seed 42) in EVERY notebook of the project -> out-of-fold (OOF) predictions of all models
# are comparable and can be stacked later without leakage.
N_FOLDS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
print('data:', DATA_DIR, '| orig exists:', ORIG_PATH.exists(), '| out:', OUT_DIR)

data: data\raw | orig exists: True | out: data\processed


## 2. Load data, preprocessing, frequency features
**What:** lower-case column names, handle the only column with missing values, separate the target, add two frequency features.

**Decisions and why**
- `arrival_delay_in_minutes` has gaps. Delays are cumulative (a late departure usually means a late arrival), so we fill gaps with `departure_delay_in_minutes` **and** keep a flag `arrival_delay_missing`, because *being* missing may itself carry signal. The filled column is then dropped as redundant with the departure delay.
- **Frequency features** are computed on train+test together. They use no labels, so they cannot leak, and they gave a measurable CV gain in `02_features_experiments`.

In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
train.columns = train.columns.str.lower().str.replace(' ', '_')
test.columns  = test.columns.str.lower().str.replace(' ', '_')

# 1. missing-value flag BEFORE fillna
train['arrival_delay_missing'] = train['arrival_delay_in_minutes'].isna().astype(int)
test['arrival_delay_missing']  = test['arrival_delay_in_minutes'].isna().astype(int)
# 2. fill gaps in arrival_delay with departure_delay
train['arrival_delay_in_minutes'] = train['arrival_delay_in_minutes'].fillna(train['departure_delay_in_minutes'])
test['arrival_delay_in_minutes']  = test['arrival_delay_in_minutes'].fillna(test['departure_delay_in_minutes'])
# 3. target
y_train = train['satisfaction'].astype(int)
# 4. drop id / target / arrival_delay
CAT_COLS  = ['gender', 'customer_type', 'type_of_travel', 'class']
DROP_COLS = ['id', 'satisfaction', 'arrival_delay_in_minutes']
X_raw      = train.drop(columns=DROP_COLS)
X_test_raw = test.drop(columns=['id', 'arrival_delay_in_minutes'])

# Frequency features: how often a value appears in train+test together. Labels are NOT used, so there is no target leakage.
def add_features(train_df, test_df):
    """flight_distance_freq and flight_distance_type_of_travel_freq (counts over train+test, no target)."""
    train_df = train_df.copy(); test_df = test_df.copy()
    full = pd.concat([train_df, test_df], ignore_index=True); n = len(train_df)
    counts = full['flight_distance'].map(full['flight_distance'].value_counts())
    train_df['flight_distance_freq'] = counts.iloc[:n].values
    test_df['flight_distance_freq']  = counts.iloc[n:].values
    key = full['flight_distance'].astype(str) + '_' + full['type_of_travel'].astype(str)
    counts = key.map(key.value_counts())
    train_df['flight_distance_type_of_travel_freq'] = counts.iloc[:n].values
    test_df['flight_distance_type_of_travel_freq']  = counts.iloc[n:].values
    return train_df, test_df

# Categorical columns become pandas "category" dtype: LightGBM/CatBoost/XGBoost use them natively.
X_train, X_test = add_features(X_raw, X_test_raw)
for col in CAT_COLS:
    X_train[col] = X_train[col].astype('category')
    X_test[col]  = X_test[col].astype('category')
print(f'Train: {X_train.shape}, Test: {X_test.shape} | positive rate {y_train.mean():.3f}')

Train: (699635, 23), Test: (299844, 23) | positive rate 0.444


## 3. Original-dataset signal (`orig_prob`, `orig_logit`)
**Idea:** the competition data was synthetically generated from a real airline survey. A model that learned the *real* relationships from the original data gives each competition row an independent "second opinion".

**How:** a deliberately regularized LightGBM is trained **only on the original dataset** and then scores the competition rows. No competition labels are involved → no target leakage, and no need for fold-wise handling.

**Why here, if `02` §17 rejected `orig_pred`?** §17 tested a first version (an untuned 300-tree model) on top of the 23 columns *without* the target encoding, and it gave −0.00008. This version is a more regularised model (stronger L2, half of the columns per tree), and the ablation in section 6 measures it on top of the final set: **+0.00011, better on all 5 folds**. Small, but consistent.

**Check:** its AUC on competition train is ≈ 0.95472 — high for a model that never saw this data, which confirms the two datasets share the same structure.

In [3]:
orig = pd.read_csv(ORIG_PATH)
orig.columns = orig.columns.str.lower().str.replace(' ', '_')
feats = [c for c in train.columns if c in orig.columns and c not in ('id', 'satisfaction', 'arrival_delay_missing')]
print(orig.shape, '| positive rate:', round(orig['satisfaction'].astype(int).mean(), 3), '| features:', len(feats))

orig_f = orig.copy()
orig_f['arrival_delay_in_minutes'] = orig_f['arrival_delay_in_minutes'].fillna(orig_f['departure_delay_in_minutes'])
orig_y = orig_f['satisfaction'].astype(int)

orig_X  = orig_f[feats].copy()
tr_orig = train[feats].copy()
te_orig = test[feats].copy()
for col in CAT_COLS:
    cats = sorted(set(orig_X[col].astype(str)) | set(tr_orig[col].astype(str)) | set(te_orig[col].astype(str)))
    for df in (orig_X, tr_orig, te_orig):
        df[col] = pd.Categorical(df[col].astype(str), categories=cats)

# Strong regularisation (reg_lambda, colsample 0.5, subsample 0.8): the original dataset is small, we want a smooth signal, not memorisation.
m_orig = lgb.LGBMClassifier(
    objective='binary', n_estimators=600, learning_rate=0.05, max_depth=8,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.5,
    reg_lambda=5.0, reg_alpha=0.05, random_state=123, n_jobs=-1, verbosity=-1)
m_orig.fit(orig_X, orig_y, categorical_feature=CAT_COLS)

# Scores for competition rows. These probabilities become features (orig_prob, orig_logit).
p_tr = m_orig.predict_proba(tr_orig)[:, 1]
p_te = m_orig.predict_proba(te_orig)[:, 1]
orig_auc = roc_auc_score(y_train, p_tr)
print('AUC of the orig-model on competition train:', round(orig_auc, 5), '(reference 0.95472)')
# logit = log(p/(1-p)). Trees do not care, but neural networks and the logistic-regression stacker work better on logits than on raw probabilities near 0/1. Clip avoids log(0).
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1 - 1e-6))

(129880, 22) | positive rate: 0.434 | features: 21


AUC of the orig-model on competition train: 0.95472 (reference 0.95472)


## 4. Route profile (`fdm_*`)
**Idea:** `flight_distance` takes a limited set of values and behaves like a *route ID*. All passengers on one route share route-level properties (class mix, delays, typical age, …).

**How:** for each distance value, take the mean of every other feature over all rows of that route (categoricals as integer codes), then attach it to every row. This describes *the route* alongside *the passenger*. Train+test are used together; no labels are involved.

**Why here, if `02` E14 rejected a profile?** E14 profiled only 3 ratings (and their deviations). This profile covers **all 20 other columns**, including the route's *composition*: share of Business class, of business travellers, of loyal customers, typical age and delays. That composition is what the ratings profile missed. Ablation in section 6: **+0.00020, better on all 5 folds**.

In [4]:
full = pd.concat([X_train, X_test], ignore_index=True)
n = len(X_train)
prof_cols = [c for c in X_train.columns
             if c not in ('flight_distance', 'flight_distance_freq', 'flight_distance_type_of_travel_freq')]
# Categoricals -> integer codes so that a mean over a route is defined (e.g. share of Business class on the route).
coded = full[prof_cols].copy()
for c in CAT_COLS:
    coded[c] = coded[c].astype('category').cat.codes
coded['flight_distance'] = full['flight_distance'].values
# transform("mean") puts the route mean back on every row of that route. Computed on train+test together; no labels involved.
grp = coded.groupby('flight_distance')
PROF = pd.DataFrame({f'fdm_{c}': grp[c].transform('mean').astype('float32') for c in prof_cols})
print('profile columns:', PROF.shape[1])

profile columns: 20


## 5. Assemble `X_final` and save
Concatenate the blocks, restore `category` dtype, run **sanity checks** (same columns in train/test, expected row counts 699 635 / 299 844, no NaN, orig-model AUC within ±0.001 of the reference) and write parquet files.

**Safe re-run.** All models of the project were trained on the saved files. If they already exist, the rebuilt table is *compared* with them instead of overwriting them (`OVERWRITE = False`), so a re-run can never silently change the inputs of finished models. If an assertion fails, the feature set no longer matches the frozen one and all downstream results would be incomparable.

In [5]:
X_final = X_train.copy()
X_test_final = X_test.copy()
for c in PROF.columns:
    X_final[c]      = PROF[c].values[:n]
    X_test_final[c] = PROF[c].values[n:]
X_final['orig_prob'], X_final['orig_logit']           = p_tr, logit(p_tr)
X_test_final['orig_prob'], X_test_final['orig_logit'] = p_te, logit(p_te)
for col in CAT_COLS:
    X_final[col]      = X_final[col].astype('category')
    X_test_final[col] = X_test_final[col].astype('category')

# Fail loudly if anything differs from the frozen reference (row counts, NaN, orig-model AUC).
# --- sanity checks ---
assert list(X_final.columns) == list(X_test_final.columns)
assert X_final.shape[0] == 699635 and X_test_final.shape[0] == 299844, 'unexpected row counts'
assert not X_final.isna().any().any() and not X_test_final.isna().any().any(), 'NaN in features'
assert abs(orig_auc - 0.95472) < 0.0010, f'orig-model AUC {orig_auc:.5f} differs from reference 0.95472'
print('X_final:', X_final.shape, '| X_test_final:', X_test_final.shape)

OVERWRITE = False      # True only if you deliberately want to replace the files all models were trained on
saved = OUT_DIR / 'X_final_train.parquet'
if saved.exists() and not OVERWRITE:
    # Compare the rebuilt table with the saved one: largest absolute difference over all numeric columns.
    old_tr, old_te = pd.read_parquet(saved), pd.read_parquet(OUT_DIR / 'X_final_test.parquet')
    num = [c for c in X_final.columns if c not in CAT_COLS]
    diff = max((X_final[num] - old_tr[num]).abs().max().max(), (X_test_final[num] - old_te[num]).abs().max().max())
    same_cats = all((X_final[c].astype(str) == old_tr[c].astype(str)).all() for c in CAT_COLS)
    print(f'saved files kept | largest numeric difference vs rebuilt: {diff:.2e} | categories identical: {same_cats}')
else:
    X_final.to_parquet(saved)
    X_test_final.to_parquet(OUT_DIR / 'X_final_test.parquet')
    y_train.reset_index(drop=True).to_frame('y').to_parquet(OUT_DIR / 'y_train.parquet')
    print('saved to', OUT_DIR, '->', sorted(p.name for p in OUT_DIR.glob('*.parquet')))

X_final: (699635, 45) | X_test_final: (299844, 45)


saved files kept | largest numeric difference vs rebuilt: 2.06e-13 | categories identical: True


## 6. Does each block earn its place? (ablation)

**Question:** the route profile and the original-data signal were added after `02`. Does each of them actually improve the model, or do they only add columns?

**Method (ablation):** start from the final set of `02` (21 baseline + 2 frequency columns + in-fold TE of the distance, CV 0.96015) and add one block at a time, then both. Everything else is fixed: the **baseline LightGBM parameters of `01`/`02`** (untuned, so the comparison measures features, not tuning), the same 5 folds, TE fitted inside each fold. The signal threshold of `02` is +0.0003, and we also look at how many folds improve.

**Runtime:** ≈ 3 minutes (4 variants × 5 folds). Set `RUN_ABLATION = False` to skip.

In [6]:
def add_te_flight_distance(X_fit, y_fit, other_frames, smooth='auto'):
    """te_flight_distance WITHOUT leakage. Call INSIDE a fold:
    X_fit/y_fit = training part of the fold; other_frames = validation part, test set."""
    enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=RANDOM_STATE)
    key = lambda d: d[['flight_distance']].astype(str)
    X_fit = X_fit.copy()
    X_fit['te_flight_distance'] = enc.fit_transform(key(X_fit), y_fit)[:, 0]
    encoded = []
    for frame in other_frames:
        frame = frame.copy()
        frame['te_flight_distance'] = enc.transform(key(frame))[:, 0]
        encoded.append(frame)
    return X_fit, encoded


RUN_ABLATION = True

BASELINE_PARAMS = dict(objective='binary', metric='auc', learning_rate=0.05, num_leaves=127, min_child_samples=20,
                       subsample=0.8, colsample_bytree=0.8, n_estimators=1000, early_stopping_rounds=50,
                       verbose=-1, random_state=RANDOM_STATE)    # exactly the parameters of 01 / 02 (untuned)

FDM_COLS = [c for c in X_final.columns if c.startswith('fdm_')]            # route profile (20 columns)
ORIG_COLS = ['orig_prob', 'orig_logit']                                    # original-data signal (2 columns)
BASE_COLS = [c for c in X_final.columns if c not in FDM_COLS + ORIG_COLS]  # the 23 columns of 02


def cv_with_te(cols):
    # 5-fold CV of LightGBM on X_final[cols] + in-fold target encoding of flight_distance.
    # Returns the OOF AUC and the five fold AUCs.
    oof, fold_auc = np.zeros(len(X_final)), []
    for tr_idx, val_idx in skf.split(X_final, y_train):
        X_tr, X_val = X_final.iloc[tr_idx][cols], X_final.iloc[val_idx][cols]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        X_tr, (X_val,) = add_te_flight_distance(X_tr, y_tr, [X_val])     # encoder fitted on the training part only
        model = lgb.LGBMClassifier(**BASELINE_PARAMS)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
        fold_auc.append(roc_auc_score(y_val, oof[val_idx]))
    return roc_auc_score(y_train, oof), np.array(fold_auc)


if RUN_ABLATION:
    variants = {'23 columns + TE (end of 02)': BASE_COLS,
                '+ route profile': BASE_COLS + FDM_COLS,
                '+ original-data signal': BASE_COLS + ORIG_COLS,
                '+ both = X_final': BASE_COLS + FDM_COLS + ORIG_COLS}
    results = {name: cv_with_te(cols) for name, cols in variants.items()}
    ref_auc, ref_folds = results['23 columns + TE (end of 02)']
    table = pd.DataFrame({name: {'CV AUC': f'{auc:.5f}', 'gain': f'{auc - ref_auc:+.5f}',
                                 'folds improved': f'{int((folds > ref_folds).sum())}/5'}
                          for name, (auc, folds) in results.items()}).T
    print(table.to_string())

                              CV AUC      gain folds improved
23 columns + TE (end of 02)  0.96015  +0.00000            0/5
+ route profile              0.96036  +0.00020            5/5
+ original-data signal       0.96027  +0.00011            5/5
+ both = X_final             0.96048  +0.00032            5/5


**What we see**

| Feature set | CV AUC | Gain | Folds improved |
|---|---|---|---|
| 23 columns + TE (end of `02`) | 0.96015 | – | – |
| + route profile | 0.96036 | +0.00020 | 5 / 5 |
| + original-data signal | 0.96027 | +0.00011 | 5 / 5 |
| + both = `X_final` | 0.96048 | **+0.00032** | 5 / 5 |

- The first row reproduces the final CV of `02` exactly (0.96015), so the comparison is consistent.
- **Each block helps on every fold.** Alone, each stays below the +0.0003 threshold; **together they clear it** (+0.00032), and their gains almost add up, so they carry different information.
- **Decision:** keep both blocks in `X_final`.

## 7. (Optional) Reproduction check — tuned LightGBM on `X_final`
Set `RUN_CHECK = True` once (~25–40 min on CPU) to confirm the rebuilt features match the earlier ones: expected **CV ≈ 0.96072** (folds 0.96095 / 0.95984 / 0.96153 / 0.96080 / 0.96057).

It uses the **in-fold target-encoding helper** `add_te_flight_distance` defined in section 6 (later notebooks reuse it): it fits the encoder on the fold's training part only (with an inner 5-fold fit, `cv=5`, so training rows are not encoded with their own label) and applies it to validation/test.

In [7]:
RUN_CHECK = False

# add_te_flight_distance is defined in section 6 (ablation).

LGBM_OPT_PARAMS = {
    'objective': 'binary', 'metric': 'auc', 'verbose': -1, 'random_state': RANDOM_STATE,
    'n_estimators': 1000, 'early_stopping_rounds': 50,
    'learning_rate': 0.011594054946533571, 'num_leaves': 232, 'min_child_samples': 69,
    'subsample': 0.9851580134791578, 'colsample_bytree': 0.6505963546420028,
    'reg_alpha': 1.5621237403785044, 'reg_lambda': 1.7672411530478345,
}

# Optional reproduction check; off by default because it takes 25-40 min.
if RUN_CHECK:
    oof = np.zeros(len(X_final))
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_final, y_train), 1):
        X_tr, X_val = X_final.iloc[tr_idx], X_final.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        X_tr, (X_val,) = add_te_flight_distance(X_tr, y_tr, [X_val])
        model = lgb.LGBMClassifier(**LGBM_OPT_PARAMS)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
        print(f'Fold {fold} AUC: {roc_auc_score(y_val, oof[val_idx]):.5f}')
    print(f'CV AUC (OOF): {roc_auc_score(y_train, oof):.5f}  (expected ~0.96072)')

## Summary & decisions
- **Result:** `X_final` = 45 columns: 21 baseline + 2 frequency + 20 route-profile + 2 original-data columns. Plus `te_flight_distance`, computed inside each fold.
- **Where each block comes from:** frequencies and TE were accepted in `02` (E8, E9, E12). The route profile and the original-data signal were added afterwards and **measured here** (section 6): +0.00020 and +0.00011, together +0.00032, better on all 5 folds.
- **Score:** untuned LightGBM 0.96048; tuned LightGBM on `X_final` CV **0.96072** (0.96088 after the final Optuna run in `04`), versus 0.95885 for the plain baseline (`01`): about **+0.0019** from features and tuning together.
- **What is intentionally not saved:** `te_flight_distance` (it uses the labels, so it is fitted inside each fold).
- **Next:** notebooks 04–07 and 11 load these files. Notebook 09 extends them into `X_rich` (92 columns) for neural networks.